# STAT 764 · Meeting 9 — Classification foundations

**Tuesday, October 6**

Since October 2012, Medicare has cut payments to hospitals whose patients come back within
30 days more often than expected — at first after heart attack, heart failure and
pneumonia, and by up to 3% of what Medicare pays the hospital for inpatient care. So
hospitals build models to decide who gets a follow-up call, a pharmacist's review, a
nurse's visit.

That model is a classifier, and a classifier is **two things, not one**: a probability for
each patient, and a rule that turns the probability into an action. A lot of what goes
wrong with classifiers goes wrong because the second thing was chosen by default.

Today, on about 100,000 real hospital encounters for patients with diabetes:

1. **Define the problem** before touching a model.
2. **A model that looks accurate** — until you ask what doing nothing scores.
3. **The probability and the rule**, pulled apart.
4. **The confusion matrix**, one threshold at a time.
5. **ROC curves and AUC** — how to read one, and what AUC can and cannot tell you.
6. **Precision depends on who you are predicting for.**
7. **The unit you predict for** — the same patient on both sides of the split. Meeting 6's
   exit ticket asked for a dataset where shuffling rows would be dishonest; three of you
   named this one. Today it is measured.

In [ ]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
found = ([p for p in [here, *here.parents] if (p / "course" / "stat764.py").exists()]
         + [c.parent.parent for c in here.glob("*/course/stat764.py")])
if os.environ.get("STAT764_REPO"):          # your clone, when it is not above you
    found.insert(0, pathlib.Path(os.environ["STAT764_REPO"]))

if found:
    sys.path.insert(0, str(found[0] / "course"))
else:
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/DataScienceUWL/stat764-fall2026"
        "/main/course/stat764.py", "stat764.py")
    sys.path.insert(0, ".")
    print("  (no local clone found — pulled the helpers from GitHub)")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from stat764 import diabetes

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score, roc_curve
from sklearn.model_selection import (GroupKFold, GroupShuffleSplit, KFold,
                                     cross_val_score, train_test_split)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

raw = diabetes()
print(f"{len(raw):,} encounters · {raw['patient_nbr'].nunique():,} patients · "
      f"{raw.shape[1]} columns\n")
print(raw["readmitted"].value_counts().to_string())

## 1. Define the problem first

**What one row is:** one hospital stay — an *encounter* — of a patient with diabetes. Ten of
the 50 columns, for six stays drawn at random:

In [ ]:
# What one row is: one hospital STAY (an "encounter"), not one patient. The same patient
# can appear many times -- patient_nbr is how you tell.
PREVIEW = ["patient_nbr", "age", "gender", "time_in_hospital", "num_medications",
           "number_inpatient", "A1Cresult", "insulin", "discharge_disposition_id",
           "readmitted"]
raw[PREVIEW].sample(6, random_state=5)          # six stays drawn at random

A few things to read off it. `age` comes in ten-year bands. `A1Cresult` is blank (`NaN`)
when the test was not done — for 83% of stays. `discharge_disposition_id` is a code: 1 is
home, 11 is death. And `readmitted`, the outcome, has three values: not readmitted (`NO`),
readmitted after more than 30 days (`>30`), and readmitted **within 30 days** (`<30`). The
last is what hospitals are measured on, so it is the thing to predict — a yes/no outcome.

One more decision before any model. Some encounters ended with the patient's death, or
with a discharge to hospice. Look at how often those patients were "readmitted":

In [ ]:
ENDED = [11, 13, 14, 19, 20, 21]        # discharge codes for death or hospice
ended = raw["discharge_disposition_id"].isin(ENDED)
print(f"  encounters ending in death or hospice    {ended.sum():,}")
print(f"  readmitted within 30 days, of those      {(raw.loc[ended, 'readmitted'] == '<30').mean():.1%}")
print(f"  readmitted within 30 days, of the rest   {(raw.loc[~ended, 'readmitted'] == '<30').mean():.1%}")

d = raw[~ended].copy()
d["readmit30"] = (d["readmitted"] == "<30").astype(int)
print(f"\n  kept: {len(d):,} encounters from {d['patient_nbr'].nunique():,} patients")

A patient who died cannot be readmitted, and a model that learns "this one will not come
back" from a death has learned nothing a nurse can act on. Those encounters come out —
not because it improves any number, but because they are not part of the question.

⚠ Notice that this was a decision, made before any model, and nothing in the code would
have made it for you.

The rows are **encounters**, not patients. Many patients appear more than once, so every
patient's encounters go on **one** side of the split. Section 7 measures whether that
mattered.

In [ ]:
NUM = ["time_in_hospital", "num_lab_procedures", "num_procedures", "num_medications",
       "number_outpatient", "number_emergency", "number_inpatient", "number_diagnoses"]
CAT = ["age", "gender", "race", "admission_type_id", "discharge_disposition_id",
       "admission_source_id", "A1Cresult", "max_glu_serum", "insulin", "change",
       "diabetesMed", "medical_specialty"]
d[CAT] = d[CAT].astype(str)            # the codes are labels, not quantities


def pipeline():
    """The pipeline since Meeting 2 -- fill gaps, scale, one-hot encode, then the model."""
    return Pipeline([
        ("prep", ColumnTransformer([
            ("num", Pipeline([("fill", SimpleImputer(strategy="median")),
                              ("scale", StandardScaler())]), NUM),
            ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), CAT),
        ])),
        ("model", LogisticRegression(max_iter=2000)),
    ])


visits = d.groupby("patient_nbr").size()
train_rows, test_rows = next(GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=764)
                             .split(d, groups=d["patient_nbr"]))
train, test = d.iloc[train_rows], d.iloc[test_rows]
X_train, y_train = train[NUM + CAT], train["readmit30"]
X_test, y_test = test[NUM + CAT], test["readmit30"]

print(f"  patients with more than one encounter   {(visits > 1).sum():,}   (one of them {visits.max()} times)")
print(f"  train {len(train):,} encounters · test {len(test):,} encounters · "
      f"patients in both: {len(set(train['patient_nbr']) & set(test['patient_nbr']))}")

## 2. Warm-up — is this a good model?

A logistic regression, fitted to the training patients and scored on encounters from
patients it has never seen.

In [ ]:
model = pipeline().fit(X_train, y_train)
print(f"  accuracy on {len(test):,} unseen encounters   {accuracy_score(y_test, model.predict(X_test)):.4f}")

Good? Before you answer: what would **doing nothing** score — predicting "not readmitted"
for every single encounter?

In [ ]:
nobody = np.zeros(len(y_test), dtype=int)
caught = int(((model.predict(X_test) == 1) & (y_test.to_numpy() == 1)).sum())

print(f"  predict 'not readmitted' for everyone   accuracy {accuracy_score(y_test, nobody):.4f}")
print(f"  the model                               accuracy {accuracy_score(y_test, model.predict(X_test)):.4f}")
print(f"\n  readmissions in the test set            {int(y_test.sum()):,}")
print(f"  readmissions the model flagged          {caught}")

**The model is slightly less accurate than doing nothing**, and it flagged 48 of the 2,739
patients who came back.

Nothing is broken. Accuracy rewards getting the common answer right, and when nearly nine
answers in ten are "no", the cheapest way to be accurate is to say "no" to everyone. A
number that scores ignoring the rare class as success cannot tell you which classifier is
better — and the rare class is the one the hospital cares about.

## 3. A classifier is a probability and a rule

`model.predict` hid two steps. Pull them apart.

In [ ]:
p = model.predict_proba(X_test)[:, 1]        # the model's estimate of P(readmitted within 30 days)

print(f"  predicted probabilities: median {np.median(p):.3f}, mean {p.mean():.3f}, "
      f"largest {p.max():.3f}")
print(f"  share of encounters above 0.5        {np.mean(p > 0.5):.2%}   ({(p > 0.5).sum()} of {len(p):,})")
print(f"  model.predict is exactly 'p > 0.5'   {np.array_equal(model.predict(X_test), (p > 0.5).astype(int))}")

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(p, bins=60, color="grey")
ax.axvline(0.5, color="black", ls="--")
ax.annotate("  the default rule: 0.5", (0.5, ax.get_ylim()[1] * 0.8))
ax.set_xlabel("predicted probability of readmission within 30 days")
ax.set_ylabel("encounters")
plt.show()

The average predicted probability is close to the share who were actually readmitted —
roughly what an honest logistic regression does. Almost nobody is above 0.5, because
almost nobody has a better-than-even chance of coming back. **The model is doing its job.**

**The rule is the problem.** 0.5 is not part of the model. scikit-learn chose it for you,
and it means "act only on patients who are more likely than not to be readmitted".
Nothing about a follow-up phone call says that. Thursday is entirely about choosing the
rule; today is about seeing what it controls.

## 4. The confusion matrix — one threshold at a time

The whole map first — the four boxes, and every rate that is built from them:

![An extended confusion matrix. Rows are what actually happened: came back, or did not. Columns are what the rule did: flagged, or not flagged. The four boxes are true positive (caught), false negative (missed, a Type II error), false positive (a false alarm, a Type I error) and true negative (rightly left alone). To the right, read across a row: sensitivity equals TP over TP plus FN, and specificity equals TN over FP plus TN, with the false negative and false positive rates. Below, read down a column: precision equals TP over TP plus FP, and negative predictive value equals TN over FN plus TN. Also prevalence and accuracy, which use all four boxes.](extended_confusion_matrix.png)

Pick a threshold and every encounter lands in one of four boxes: what actually happened,
against what the rule did. Here are the four boxes at the default rule, 0.5:

In [ ]:
# A threshold turns probabilities into decisions. The confusion matrix counts the decisions.
flag = p >= 0.5                                  # the default rule: flag if p is at least 0.5

# Rows: what actually happened. Columns: what the rule did. pd.crosstab counts every pair,
# which gives the four boxes with their names on them.
actual = pd.Series(np.where(y_test == 1, "came back", "did not come back"), name="actually")
rule = pd.Series(np.where(flag, "flagged", "not flagged"), name="the rule at 0.5")
print(pd.crosstab(actual, rule).loc[["came back", "did not come back"], ["flagged", "not flagged"]])

| | **flagged** | **not flagged** |
|---|---|---|
| **came back** | **true positive (TP)** — caught | **false negative (FN)** — missed |
| **did not come back** | **false positive (FP)** — a false alarm | **true negative (TN)** — rightly left alone |

Two kinds of error, and they are not the same mistake. A **false alarm** spends a nurse's
call on someone who was fine. A **miss** sends home a patient who comes back. You have met
both before: make "this patient will not come back" the null hypothesis and flagging the
rejection, and a false alarm is a **Type I error**, a miss is a **Type II error**, and the
share of real readmissions you flag is the test's **power**.

Every rate people quote is a ratio of these four boxes, and two fields named them twice —
medicine says *sensitivity*, machine learning and scikit-learn say *recall*:

| name | also called | from the boxes | in words |
|---|---|---|---|
| **sensitivity** | recall, true positive rate; *power* | TP / (TP + FN) | of those who **came back**, the share you flagged |
| **specificity** | true negative rate | TN / (TN + FP) | of those who **did not**, the share you left alone |
| **false positive rate** | 1 − specificity; *the Type I error rate* | FP / (FP + TN) | of those who did not, the share you flagged anyway |
| **false negative rate** | 1 − sensitivity; *the Type II error rate* | FN / (FN + TP) | of those who came back, the share you missed |
| **precision** | positive predictive value (PPV) | TP / (TP + FP) | of those you **flagged**, the share who came back |
| **negative predictive value** | NPV | TN / (TN + FN) | of those you **did not flag**, the share who did not come back |
| **accuracy** | | (TP + TN) / everyone | the share the rule got right |

For this problem, at the default rule — the four boxes above:

- **Sensitivity** — of the 2,739 patients who came back within 30 days, the rule flagged 48:
  **1.8%**. A nurse working from this list would never hear about 98 of every 100
  readmissions.
- **Specificity** — of the 21,967 who did not come back, the rule left 21,913 alone:
  **99.8%**. Almost no false alarms — because it almost never flags anyone.
- **Precision** — of the 102 patients it flagged, 48 came back: **47%**. When the rule does
  speak it is right about half the time; it just speaks for 102 of 24,706 patients.

Sensitivity and specificity read **across a row**: they start from what happened.
Precision and NPV read **down a column**: they start from what you did. That one difference
is why Section 6 exists.

Now the same probabilities under three rules:

In [ ]:
# The model never changes below -- only the threshold does. Each row is one rule.
rows = []
for threshold in (0.5, 0.2, 0.11):
    flag = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y_test, flag).ravel()   # the four boxes, as numbers
    rows.append({"threshold": threshold, "flagged": flag.mean(),
                 "TP": tp, "FP": fp, "FN": fn, "TN": tn,
                 "sensitivity": tp / (tp + fn), "specificity": tn / (tn + fp),
                 "precision": tp / (tp + fp), "accuracy": (tp + tn) / len(flag)})
print(pd.DataFrame(rows).round(3).to_string(index=False))

Move the threshold from 0.5 to 0.11 and sensitivity climbs from 0.018 to 0.572, while
precision falls from 0.471 to 0.168 and accuracy from 0.889 to 0.639.

**Every number in that table moved. The model did not.** Which row is right depends on
what a flag costs and what a miss costs — and those are facts about the hospital, not the
data.

### From the lab — the same four boxes, in breast cancer

My research group works with Mayo Clinic Health System on deep-learning models that read
breast ultrasound exams and score how likely a lesion is to be cancer. One of them, on 12,357
exams from 2018–2024 that it never saw during training: 1,004 cancers (8%), AUC 0.93. Saying
"benign" to everyone would be 92% accurate.

At a threshold that catches 95% of the cancers, per 1,000 women scanned (rounded):

| | **model flags** | **model clears** |
|---|---|---|
| **cancer** | 77 caught | **4 missed** — a Type II error |
| **no cancer** | **442 flagged** — a Type I error: a biopsy, a wait, a frightening week | 476 rightly cleared |

The same model at other thresholds, per 1,000 women:

| threshold catches | cancers caught | cancers missed | healthy women flagged |
|---|---|---|---|
| 99% of cancers | 80 | 1 | 776 |
| 95% | 77 | 4 | 442 |
| 90% | 73 | 8 | 254 |
| 80% | 65 | 16 | 51 |
| 70% | 57 | 24 | 14 |

The two errors are not the same mistake. A false alarm costs a biopsy; a miss can cost a
delayed diagnosis. So the clinic lives near the top of that table and accepts a great many
false alarms — and this model only advises: a radiologist still reads every exam. Where to
stop is Thursday's question.

## 5. ROC curves and AUC

### Reading an ROC curve

Every threshold gives one confusion matrix, and from it two of Section 4's rates:
**sensitivity**, the share of readmitted patients you flagged, and the **false positive
rate** (1 − specificity), the share of the others you flagged anyway. Plot one against the
other and a threshold becomes a single **point**. Section 4's three thresholds are three
points:

| threshold | false positive rate — across | sensitivity — up |
|---|---|---|
| 0.5 | 0.002 | 0.018 |
| 0.2 | 0.070 | 0.193 |
| 0.11 | 0.353 | 0.572 |

Slide the threshold from high to low and the point travels from (0, 0) — nobody flagged —
to (1, 1) — everybody flagged. The path it traces is the **ROC curve**. (The name comes from
Second World War radar: *receiver operating characteristic*.)

- The **diagonal** is a coin: flagging at random catches readmissions exactly as fast as
  it raises false alarms.
- A good model **bows toward the top-left corner**: working down its list, it catches
  readmissions faster than it raises false alarms.
- The **top-left corner** — every readmission caught, no false alarms — is a perfect model.
- **AUC** is the area under the curve: 1 for perfect, 0.5 for a coin.

Here is the whole idea on ten made-up patients:

In [ ]:
# ROC by hand: ten made-up patients, sorted from highest probability to lowest.
toy = pd.DataFrame({"probability": [0.92, 0.81, 0.74, 0.66, 0.55, 0.43, 0.35, 0.28, 0.17, 0.08],
                    "came back":   [1,    1,    0,    1,    0,    0,    1,    0,    0,    0]})
came, did_not_come = toy["came back"].sum(), (1 - toy["came back"]).sum()   # 4 and 6

# Lower the threshold past one patient at a time. A patient who came back moves the point
# UP by 1/4 (one more caught); one who did not moves it RIGHT by 1/6 (one more false alarm).
sens_toy = np.r_[0, toy["came back"].cumsum() / came]
fpr_toy = np.r_[0, (1 - toy["came back"]).cumsum() / did_not_come]

# AUC two ways: the area under the staircase, and the share of (came back, did not) pairs
# that the ranking puts in the right order.
area = np.sum(np.diff(fpr_toy) * sens_toy[1:])
pairs = [(a, b) for a in toy.loc[toy["came back"] == 1, "probability"]
                for b in toy.loc[toy["came back"] == 0, "probability"]]
right_order = sum(a > b for a, b in pairs)
print(f"  area under the staircase                          {area:.3f}")
print(f"  pairs in the right order: {right_order} of {len(pairs)}                    {right_order / len(pairs):.3f}")

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr_toy, sens_toy, marker="o", color="tab:blue", label=f"the ten patients, AUC {area:.2f}")
ax.fill_between(fpr_toy, sens_toy, color="tab:blue", alpha=0.12)
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="a coin, AUC 0.5")
for x, y, prob in zip(fpr_toy[1:], sens_toy[1:], toy["probability"]):
    ax.annotate(f" {prob}", (x, y), fontsize=8)            # the threshold that reaches each point
ax.set_xlabel("false positive rate (one step right = one false alarm)")
ax.set_ylabel("sensitivity (one step up = one readmission caught)")
ax.legend(loc="lower right")
plt.show()

Four of the ten came back and six did not. Every step up is a readmission caught; every
step right is a false alarm. A model that ranked all four readmissions first would climb
straight to the top before moving right — the corner, area 1. A coin wanders along the
diagonal, area 0.5. This one climbs mostly first: of the 24 pairs of one patient who came back
and one who did not, it puts **20 in the right order, and the area under its staircase is
20 / 24 = 0.83**. That is what AUC is: the share of such pairs the model ranks correctly.

### AUC — a ranking, not a rate

Now the real model, and one number that does not depend on any threshold:

In [ ]:
auc = roc_auc_score(y_test, p)

rng = np.random.default_rng(764)
came_back = rng.choice(p[y_test.to_numpy() == 1], 100_000)
did_not = rng.choice(p[y_test.to_numpy() == 0], 100_000)

print(f"  AUC                                                          {auc:.4f}")
print(f"  100,000 random pairs (one readmitted, one not): the readmitted")
print(f"  patient got the higher probability in                        {np.mean(came_back > did_not):.4f}")

fpr, tpr, cut = roc_curve(y_test, p)
fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, label=f"logistic regression, AUC {auc:.3f}")
ax.plot([0, 1], [0, 1], ls="--", color="grey", label="a coin, AUC 0.5")
for t in (0.5, 0.2, 0.11):
    k = np.argmin(np.abs(cut - t))
    ax.plot(fpr[k], tpr[k], "o", color="black")
    ax.annotate(f"  threshold {t}", (fpr[k], tpr[k]))
ax.set_xlabel("1 − specificity (share of the not-readmitted you flagged)")
ax.set_ylabel("sensitivity (share of the readmitted you flagged)")
ax.legend(loc="lower right")
plt.show()

**AUC is a statement about ranking.** Pick one patient who was readmitted and one who was
not: the model gives the readmitted one the higher probability about 65% of the time. A
coin does it 50% of the time.

The curve is the confusion matrix at **every** threshold at once — each of Section 4's
three rows is one dot on it. That is AUC's strength and its limit:

- it cannot tell you **which** threshold to use, and
- it does not care whether the probabilities are **right**, only whether they are in the
  right order. Meeting 11, on calibration, is about whether they are right.

## 6. Precision depends on who you are predicting for

Some patients appear once in these data; some appear five, ten, forty times — and the
more often a patient appears, the more often they were readmitted. (You cannot know at
discharge how many more times someone will come back, so this is a fact about the
population, not a column to predict with.)

Take the model's sensitivity and specificity at threshold 0.2, and ask what its precision
**would** be in populations with those different base rates:

In [ ]:
flag = p >= 0.2
sens = flag[y_test.to_numpy() == 1].mean()
spec = (~flag)[y_test.to_numpy() == 0].mean()

times = d.groupby("patient_nbr")["readmit30"].transform("size").clip(upper=5)
base_rate = d.groupby(times)["readmit30"].mean()

print(f"  threshold 0.2: sensitivity {sens:.3f}, specificity {spec:.3f}\n")
print("  patients who appear      readmitted   precision, if sensitivity and specificity held")
for k, rate in base_rate.items():
    precision = sens * rate / (sens * rate + (1 - spec) * (1 - rate))
    label = "5 or more times" if k == 5 else f"{k} time{'s' if k > 1 else ''}"
    print(f"  {label:<24} {rate:>9.1%}   {precision:>9.1%}")

**Same sensitivity, same specificity — and precision runs from about 11% to about 52%.**

Precision is not a property of the model. It is a property of the model **and the
population you point it at**. Flag the same way where readmission is rare and most of your
flags are false alarms. That is Bayes' theorem, and it is why a test that is 93% specific
can still be wrong about most of the people it flags.

⚠ That arithmetic **assumed** the model is equally sensitive and specific in every group.
Studio question 2 checks whether that assumption holds.

### From the lab — one model, three populations

The breast ultrasound model at the threshold that catches 90% of cancers (specificity 72%).
Hold both fixed and change only who walks in:

| population | share with cancer | precision (PPV) | one cancer per … women flagged |
|---|---|---|---|
| screening everyone | about 0.5% | 2% | 62 |
| diagnostic ultrasound — this test set | 8% | 22% | 4.5 |
| BI-RADS 4 referrals | about 30% | 58% | 1.7 |

The same flag means a 2% chance of cancer or a 58% chance, depending on the population. (The
middle row is measured; the other two prevalences are illustrative round figures.)

## 7. The unit you predict for

Back in Meeting 6, on comparing models honestly, the exit ticket asked you to name a dataset
from your own field where shuffling rows into folds would be dishonest. That meeting's own
example was time — bike rentals, where a shuffled fold trains on the future. Three of you
named a different failure the notebook never showed: **the same unit on both sides of the
split** — patient visits, a borrower's loan rows, a fish's movements. Here it is, measured.
Five folds shuffled by encounter, against five folds that keep each patient on one side:

In [ ]:
X_all, y_all, patient = d[NUM + CAT], d["readmit30"], d["patient_nbr"]

shuffled = cross_val_score(pipeline(), X_all, y_all, scoring="roc_auc",
                           cv=KFold(5, shuffle=True, random_state=0))
grouped = cross_val_score(pipeline(), X_all, y_all, scoring="roc_auc",
                          cv=GroupKFold(5), groups=patient)

print(f"  shuffled folds (a patient can be on both sides)  AUC {shuffled.mean():.4f}   "
      f"fold-to-fold SD {shuffled.std(ddof=1):.4f}")
print(f"  grouped folds  (each patient on one side)        AUC {grouped.mean():.4f}   "
      f"fold-to-fold SD {grouped.std(ddof=1):.4f}")
print(f"  difference                                           {shuffled.mean() - grouped.mean():+.4f}")

Four ten-thousandths of AUC, against folds that move by about three thousandths on their
own. **Here, measured, the leak costs nothing we can see.** A logistic regression on these
columns has no way to memorize a patient.

That is a finding — the third time this course has checked for a problem and found it
absent — and "I checked" is the sentence that earns it. It is not a reason to skip the
check on the next dataset, where a model *can* memorize the unit.

But look at **who** ends up in a shuffled test set:

In [ ]:
rand_train, rand_test = train_test_split(d, test_size=0.25, random_state=764)
returning = rand_test["patient_nbr"].isin(set(rand_train["patient_nbr"]))

print(f"  a shuffled split: {returning.mean():.1%} of test encounters come from patients "
      f"who are also in training")
print(f"  readmitted within 30 days:   returning patients {rand_test.loc[returning, 'readmit30'].mean():.1%}"
      f"    new patients {rand_test.loc[~returning, 'readmit30'].mean():.1%}")

# Why: is a readmission usually itself another row? Does the patient have a LATER encounter?
later = d["encounter_id"] < d.groupby("patient_nbr")["encounter_id"].transform("max")
print(f"\n  patient has a later encounter in the data:   readmitted within 30 days "
      f"{later[d['readmit30'] == 1].mean():.1%}    not readmitted at all "
      f"{later[d['readmitted'] == 'NO'].mean():.1%}")

Two different populations, one readmitted more than three times as often as the other —
partly because a readmission within 30 days is usually **itself another row in this
table**: 69% of those encounters have a later one for the same patient, against 1% of
encounters that were never followed by a readmission. So "how good is the model?" has different answers for a patient's first visit
and for anyone who walks in, because the base rate is different, and Section 6 showed
what a base rate does to precision.

⚠ **Decide the unit you predict for before you decide anything else.** Studio question 3
rebuilds the model on one row per patient.

### From the lab — what counts as one prediction?

In the breast ultrasound data the unit question has two candidate answers: an **image**, or
an **exam** — every image of one breast from one visit. We predict once per exam — a bag of
about five images — and split by **patient**: 70% of patients to training, 20% to validation,
10% to test, so every exam from one patient sits on one side (Hansen et al., *Journal of
Imaging Informatics in Medicine*, 2026).

Split by image instead and the leak is not subtle. **29% of the 756,315 images are
near-duplicate pairs** — the same view saved twice, with and without the sonographer's
measurement marks. Put one twin in training and the other in the test set, and the "unseen"
test image is one the model has already seen.

## Studio

The rest of class, until the exit ticket. Use your own seed for the patient-grouped split —
change `764` below. Fast or slow is fine: the notebook is due 24 hours after class, so finish
at home if you need to. Done early? Start Lab 5 Part 1 — same data, same split.

1. **Your baseline, your threshold.** Fit the pipeline on your split. Report AUC, the
   confusion matrix at 0.5, and the confusion matrix at a threshold **you** choose — with
   one sentence on why you chose it. There is no right answer yet; Thursday is about
   finding one.
2. **Does precision follow the base rate?** Section 6 assumed the model is equally
   sensitive and specific in every group. Check it: at your threshold, split your test
   encounters by prior inpatient stays — `number_inpatient == 0` against `>= 1` — and
   report prevalence, sensitivity, specificity and precision in each.
   **Before you run it, write down what Section 6's arithmetic predicts.**
3. **One row per patient.** `diabetes(patient_level=True)` keeps each patient's first
   encounter. Remove death and hospice the same way, split, fit, and report prevalence,
   AUC, and precision at your threshold. Which question — any encounter, or a patient's
   first — would a hospital actually be asking?

⚠ Question 2 is the one to finish.

In [ ]:
seed = 764                                   # <- your own number here
# GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed).split(d, groups=d["patient_nbr"])

In [ ]:
# YOUR CODE HERE — Question 1: your split, AUC, two confusion matrices

In [ ]:
# YOUR CODE HERE — Question 2: prior inpatient stays, 0 against 1 or more

In [ ]:
# YOUR CODE HERE — Question 3: one row per patient

## Compare — at the start of Thursday's class

We compare answers at the start of Thursday's class, built from the notebooks you upload. So
leave these in your notebook, in words or as printed numbers:

1. Your threshold, and your one-sentence reason for it.
2. Question 2: what Section 6's arithmetic predicted — written **before** you ran it — and
   what happened.
3. Question 3: your prevalence and AUC, and which question you think is the hospital's.

## Exit ticket

> A discharge nurse can call twenty patients a day. In one sentence, tell the nurse what
> AUC 0.655 means for that list — and name one thing the nurse needs that AUC cannot say.

**Answer in the cell below — two or three sentences — then save this notebook and upload
it to Canvas under "Studio notebook — Day 9". Your two studio points need this
answered: a blank exit ticket earns zero. It is not graded for being right — it is how
I find out what landed.**

### Your exit ticket

*(double-click to edit, and replace this line)*

## Also in this neighborhood

📗 **ISLP §4.4–4.5 — LDA, QDA, naive Bayes.** Skipped on purpose. They are other ways to
produce the probability; everything today about the rule and the evaluation applies to
them unchanged.

📗 **Precision–recall curves** (`precision_recall_curve`, `average_precision_score`). The
ROC curve's cousin that keeps the base rate in view — which is exactly what AUC leaves out.

🎓 **`class_weight="balanced"`.** A common suggestion for rare outcomes. It changes the
model's probabilities, not just where you cut them — Meeting 11, on calibration, is about
whether probabilities are right, and this is one way to make them wrong.

🚫 **Accuracy as the headline number for a rare outcome.** Section 2.